In [1]:
from pprint import pprint
from math import pi
from collections import Counter
import cirq_superstaq as css
import cirq
import warnings
import sys
sys.path.append('../')
import resource_estimation as res
from scripts.cultivate_json import GATE2STR
import scripts.layout_figures as lfs

category_colors = {
    "None": "#ffffff",
    "T Factory": "#4ddef1",
    "S Factory": "#fcc084",
    "Data Qubit": "#a4f0c2",
    "Ancilla Patch": "#f5bad6",
    "Distillation": "#E6E6FA",
    "CNOT": "#ed6340"
}

In [2]:
qubits = cirq.LineQubit.range(15) + [cirq.NamedQubit("F")]
cults = [cirq.NamedQubit(f"C{i}") for i in range(15)]
circuit = cirq.Circuit([
    cirq.ResetChannel().on_each(*qubits),
    res.lattice_surgery_primitives.Cultivate(pi/4).on_each(*cults),
    # css.Barrier(16).on(*qubits),
    cirq.H(qubits[0]),
    cirq.H(qubits[1]),
    cirq.H(qubits[3]),
    cirq.H(qubits[7]),
    cirq.H(qubits[15]),
    css.Barrier(31).on(*(qubits+cults)),
    cirq.CNOT(qubits[15], qubits[14]),
    # css.Barrier(16).on(*qubits),
    cirq.CNOT(qubits[7], qubits[8]),
    cirq.CNOT(qubits[7], qubits[9]),
    cirq.CNOT(qubits[7], qubits[10]),
    cirq.CNOT(qubits[7], qubits[11]),
    cirq.CNOT(qubits[7], qubits[12]),
    cirq.CNOT(qubits[7], qubits[13]),
    cirq.CNOT(qubits[7], qubits[14]),
    css.Barrier(31).on(*(qubits+cults)),
    cirq.CNOT(qubits[3], qubits[4]),
    cirq.CNOT(qubits[3], qubits[5]),
    cirq.CNOT(qubits[3], qubits[6]),
    cirq.CNOT(qubits[3], qubits[11]),
    cirq.CNOT(qubits[3], qubits[12]),
    cirq.CNOT(qubits[3], qubits[13]),
    cirq.CNOT(qubits[3], qubits[14]),
    css.Barrier(31).on(*(qubits+cults)),
    cirq.CNOT(qubits[1], qubits[2]),
    cirq.CNOT(qubits[1], qubits[5]),
    cirq.CNOT(qubits[1], qubits[6]),
    cirq.CNOT(qubits[1], qubits[9]),
    cirq.CNOT(qubits[1], qubits[10]),
    cirq.CNOT(qubits[1], qubits[13]),
    cirq.CNOT(qubits[1], qubits[14]),
    css.Barrier(31).on(*(qubits+cults)),
    cirq.CNOT(qubits[0], qubits[2]),
    cirq.CNOT(qubits[0], qubits[4]),
    cirq.CNOT(qubits[0], qubits[6]),
    cirq.CNOT(qubits[0], qubits[8]),
    cirq.CNOT(qubits[0], qubits[10]),
    cirq.CNOT(qubits[0], qubits[12]),
    cirq.CNOT(qubits[0], qubits[14]),
    css.Barrier(31).on(*(qubits+cults)),
    cirq.CNOT(qubits[14], qubits[2]),
    cirq.CNOT(qubits[14], qubits[4]),
    cirq.CNOT(qubits[14], qubits[5]),
    cirq.CNOT(qubits[14], qubits[8]),
    cirq.CNOT(qubits[14], qubits[9]),
    cirq.CNOT(qubits[14], qubits[11]),
    css.Barrier(31).on(*(qubits+cults)),
    # cirq.T.on_each(*qubits[:-1]),
    # css.Barrier(16).on(*qubits),
    # cirq.H.on_each(*qubits[:-1]),
    # cirq.MeasurementGate(1).on_each(*qubits[:-1]),
])
# circuit.append(cirq.Moment(cirq.T.on_each(*qubits[:-1])))
circuit.append(
    cirq.CNOT.on(ctrl, trgt) for ctrl, trgt in zip(cults, qubits[:-1])
)
circuit.append(cirq.Moment(cirq.measure_each(*cults[:-1])))
circuit.append(cirq.Moment(
    # cirq.S.controlled().on(ctrl,
    # trgt) for ctrl, trgt in zip(cults, qubits[:-1])
    cirq.S.on_each(*qubits[:-1])
))
css.Barrier(31).on(*(qubits+cults)),
circuit.append(cirq.Moment(cirq.H.on_each(*qubits[:-1])))
circuit.append(cirq.Moment(cirq.measure_each(*qubits[:-1])))
display(circuit)
print(f"Moments: {len(circuit) - sum(1 for op in circuit.all_operations() if type(op.gate) is css.Barrier)}")

┌───────────────┐
0: ─────R─────────────H───│───────────────────────────────│───────────────────────────────│───────────────────────────────│───@───@───@───@───@───@───@───│───────────────────────────│────X──────────────────────S───H───M───
                          │                               │                               │                               │   │   │   │   │   │   │   │   │                           │    │
1: ─────R─────────────H───│───────────────────────────────│───────────────────────────────│───@───@───@───@───@───@───@───│───┼───┼───┼───┼───┼───┼───┼───│───────────────────────────│────┼X─────────────────────S───H───M───
                          │                               │                               │   │   │   │   │   │   │   │   │   │   │   │   │   │   │   │   │                           │    ││
2: ─────R─────────────────│───────────────────────────────│───────────────────────────────│───X───┼───┼───┼───┼───┼───┼───│───X───┼───┼───┼───┼───┼───┼───│───X───────────────────────│────┼┼X────────────────────S───H───M───
                          │                               │                               │       │   │   │   │   │   │   │       │   │   │   │   │   │   │   │                       │    │││
3: ─────R─────────────H───│───────────────────────────────│───@───@───@───@───@───@───@───│───────┼───┼───┼───┼───┼───┼───│───────┼───┼───┼───┼───┼───┼───│───┼───────────────────────│────┼┼┼X───────────────────S───H───M───
                          │                               │   │   │   │   │   │   │   │   │       │   │   │   │   │   │   │       │   │   │   │   │   │   │   │                       │    ││││
4: ─────R─────────────────│───────────────────────────────│───X───┼───┼───┼───┼───┼───┼───│───────┼───┼───┼───┼───┼───┼───│───────X───┼───┼───┼───┼───┼───│───┼───X───────────────────│────┼┼┼┼X──────────────────S───H───M───
                          │                               │       │   │   │   │   │   │   │       │   │   │   │   │   │   │           │   │   │   │   │   │   │   │                   │    │││││
5: ─────R─────────────────│───────────────────────────────│───────X───┼───┼───┼───┼───┼───│───────X───┼───┼───┼───┼───┼───│───────────┼───┼───┼───┼───┼───│───┼───┼───X───────────────│────┼┼┼┼┼X─────────────────S───H───M───
                          │                               │           │   │   │   │   │   │           │   │   │   │   │   │           │   │   │   │   │   │   │   │   │               │    ││││││
6: ─────R─────────────────│───────────────────────────────│───────────X───┼───┼───┼───┼───│───────────X───┼───┼───┼───┼───│───────────X───┼───┼───┼───┼───│───┼───┼───┼───────────────│────┼┼┼┼┼┼X────────────────S───H───M───
                          │                               │               │   │   │   │   │               │   │   │   │   │               │   │   │   │   │   │   │   │               │    │││││││
7: ─────R─────────────H───│───@───@───@───@───@───@───@───│───────────────┼───┼───┼───┼───│───────────────┼───┼───┼───┼───│───────────────┼───┼───┼───┼───│───┼───┼───┼───────────────│────┼┼┼┼┼┼┼X───────────────S───H───M───
                          │   │   │   │   │   │   │   │   │               │   │   │   │   │               │   │   │   │   │               │   │   │   │   │   │   │   │               │    ││││││││
8: ─────R─────────────────│───X───┼───┼───┼───┼───┼───┼───│───────────────┼───┼───┼───┼───│───────────────┼───┼───┼───┼───│───────────────X───┼───┼───┼───│───┼───┼───┼───X───────────│────┼┼┼┼┼┼┼┼X──────────────S───H───M───
                          │       │   │   │   │   │   │   │               │   │   │   │   │               │   │   │   │   │                   │   │   │   │   │   │   │   │           │    │││││││││
9: ─────R─────────────────│───────X───┼───┼───┼───┼───┼───│───────────────┼───┼───┼───┼───│───────────────X───┼───┼───┼───│───────────────────┼───┼───┼───│───┼───┼───┼───┼───X───────│────┼┼┼┼┼┼┼┼┼X─────────────S───H───M───
               

Moments: 41


In [3]:
exp = cirq.Circuit([
    cirq.ResetChannel().on_each(*qubits),
    res.lattice_surgery_primitives.Cultivate(pi/4).on_each(*cults),
    # css.Barrier(16).on(*qubits),
    cirq.H(qubits[0]),
    cirq.H(qubits[1]),
    cirq.H(qubits[3]),
    cirq.H(qubits[7]),
    cirq.H(qubits[15]),
    # css.Barrier(31).on(*(qubits+cults)),

    cirq.CNOT.on(qubits[7], qubits[14]),
    cirq.CNOT.on(qubits[3], qubits[12]),
    cirq.CNOT.on(qubits[1], qubits[10]),
    cirq.CNOT.on(qubits[0], qubits[6]),
    
    cirq.CNOT.on(qubits[7], qubits[13]),
    cirq.CNOT.on(qubits[3], qubits[11]),
    cirq.CNOT.on(qubits[1], qubits[14]),
    cirq.CNOT.on(qubits[0], qubits[10]),

    cirq.CNOT.on(qubits[7], qubits[12]),
    cirq.CNOT.on(qubits[3], qubits[6]),
    cirq.CNOT.on(qubits[1], qubits[13]),
    cirq.CNOT.on(qubits[0], qubits[14]),

    cirq.CNOT.on(qubits[7], qubits[8]),
    cirq.CNOT.on(qubits[3], qubits[14]),
    cirq.CNOT.on(qubits[1], qubits[9]),
    cirq.CNOT.on(qubits[0], qubits[4]),

    cirq.CNOT.on(qubits[7], qubits[9]),
    cirq.CNOT.on(qubits[3], qubits[4]),
    cirq.CNOT.on(qubits[1], qubits[2]),
    cirq.CNOT.on(qubits[0], qubits[8]),
    cirq.CNOT.on(qubits[-1], qubits[14]),
    cirq.CNOT.on(qubits[14], qubits[11]),

    cirq.CNOT.on(qubits[7], qubits[10]),
    cirq.CNOT.on(qubits[3], qubits[5]),
    cirq.CNOT.on(qubits[1], qubits[6]),
    cirq.CNOT.on(qubits[0], qubits[12]),
    cirq.CNOT.on(qubits[14], qubits[5]),


    cirq.CNOT.on(qubits[7], qubits[11]),
    cirq.CNOT.on(qubits[3], qubits[13]),
    cirq.CNOT.on(qubits[1], qubits[5]),
    cirq.CNOT.on(qubits[0], qubits[2]),

    cirq.CNOT.on(qubits[14], qubits[9]),
    cirq.CNOT.on(qubits[14], qubits[8]),
    cirq.CNOT.on(qubits[14], qubits[4]),
    cirq.CNOT.on(qubits[14], qubits[2]),
    # css.Barrier(31).on(*(qubits+cults)),

])
# exp.append(cirq.Moment(cirq.T.on_each(*qubits[:-1])))
exp.append(
    cirq.CNOT.on(ctrl, trgt) for ctrl, trgt in zip(cults, qubits[:-1])
)
exp.append(cirq.Moment(cirq.measure_each(*cults[:-1])))
exp.append(cirq.Moment(
    # cirq.S.controlled().on(ctrl,
    # trgt) for ctrl, trgt in zip(cults, qubits[:-1])
    cirq.S.on_each(*qubits[:-1])
))
exp.append(cirq.Moment(cirq.H.on_each(*qubits[:-1])))
exp.append(cirq.Moment(cirq.measure_each(*qubits[:-1])))
display(exp)
print(f"Moments: {len(exp) - sum(1 for op in exp.all_operations() if type(op.gate) is css.Barrier)}")

┌───┐   ┌────┐   ┌───┐   ┌───┐   ┌──┐   ┌───┐   ┌─────┐   ┌──────┐   ┌────┐   ┌──┐   ┌──┐   ┌──┐
0: ─────R─────────────H────@──────────@───────@─────@────────@───────@─────@──────────X──────────────────────────────────────────S───H───M───
                           │          │       │     │        │       │     │          │
1: ─────R─────────────H────┼─@───────@┼──────@┼─────┼─@─────@┼──────@┼─────┼─────────@┼──────────X───────────────────────────────S───H───M───
                           │ │       ││      ││     │ │     ││      ││     │         ││          │
2: ─────R──────────────────┼─┼───────┼┼──────┼┼─────┼─┼─────X┼──────┼┼─────X─────────┼┼──────────┼──────────────X──────X─────────S───H───M───
                           │ │       ││      ││     │ │      │      ││               ││          │              │      │
3: ─────R─────────────H────┼@┼──────@┼┼─────@┼┼─────┼@┼─────@┼─────@┼┼───────@───────┼┼X─────────┼──────────────┼──────┼─────────S───H───M───
                           │││      │││     │││     │││     ││     │││       │       │││         │              │      │
4: ─────R──────────────────┼┼┼──────┼┼┼─────┼┼┼─────X┼┼─────X┼─────┼┼┼───────┼───────┼┼┼─────────┼───────X──────┼X─────┼─────────S───H───M───
                           │││      │││     │││      ││      │     │││       │       │││         │       │      ││     │
5: ─────R──────────────────┼┼┼──────┼┼┼─────┼┼┼──────┼┼──────┼─────X┼┼─────X─┼───────X┼┼─────────┼X──────┼──────┼┼─────┼─────────S───H───M───
                           │││      │││     │││      ││      │      ││     │ │        ││         ││      │      ││     │
6: ─────R──────────────────X┼┼──────┼┼┼─────X┼┼──────┼┼──────┼──────X┼─────┼─┼X───────┼┼─────────┼┼──────┼──────┼┼─────┼─────────S───H───M───
                            ││      │││      ││      ││      │       │     │ ││       ││         ││      │      ││     │
7: ─────R─────────────H────@┼┼─────@┼┼┼─────@┼┼─────@┼┼─────@┼─────@─┼─────┼@┼┼───────┼┼X────────┼┼──────┼──────┼┼─────┼─────────S───H───M───
                           │││     ││││     │││     │││     ││     │ │     ││││       │││        ││      │      ││     │
8: ─────R──────────────────┼┼┼─────┼┼┼┼─────┼┼┼─────X┼┼─────┼X─────┼─┼─────┼┼┼┼───────┼┼┼───────X┼┼──────┼X─────┼┼─────┼─────────S───H───M───
                           │││     ││││     │││      ││     │      │ │     ││││       │││       │││      ││     ││     │
9: ─────R──────────────────┼┼┼─────┼┼┼┼─────┼┼┼──────┼X─────X──────┼─┼─────┼┼┼┼──────X┼┼┼───────┼┼┼X─────┼┼─────┼┼─────┼─────────S───H───M───
                           │││     ││││     │││      │             │ │     ││││      ││││       ││││     ││     ││     │
10: ────R──────────────────┼┼X─────┼┼┼X─────┼┼┼──────┼─────────────X─┼─────┼┼┼┼X─────┼┼┼┼───────┼┼┼┼─────┼┼─────┼┼─────┼─────────S───H───M───
                           ││      │││      │││      │               │     │││││     ││││       ││││     ││     ││     │
11: ────R──────────────────┼┼──────┼X┼──────┼┼┼──────┼─────────────X─┼─────┼X┼┼┼─────┼┼┼┼X──────┼┼┼┼─────┼┼─────┼┼─────┼─────────S───H───M───
                           ││      │ │      │││      │             │ │     │ │││     │││││      ││││     ││     ││     │
12: ────R──────────────────┼X──────┼─┼──────X┼┼──────┼─────────────┼─X─────┼X┼┼┼─────┼┼┼┼┼──────┼┼┼┼─────┼┼─────┼┼─────┼─────────S───H───M───
                           │       │ │       ││      │             │       │││││     │││││      ││││     ││     ││     │
13: ────R──────────────────┼───────X─┼───────X┼──────┼─────────────┼───────┼┼X┼┼─────┼┼┼┼┼X─────┼┼┼┼─────┼┼─────┼┼─────┼─────────S───H───M───
                           │         │        │      │             │       ││ ││     ││││││     ││││     ││     ││     │
14: ────R──────────────────X─────────X────────X──────X──────X──────@───────@┼─┼┼─────@┼┼┼┼┼─────@┼┼┼─────@┼─────@┼─────┼X────────S───H───M───
                                                            │               │ ││      │││││      │││      │      │     ││
C0: ────CULT

Moments: 18


In [4]:
qmap = {qubits[-1]: cirq.GridQubit(7, 2)}
for idx, (q, f) in enumerate(zip(qubits, cults)):
    row = idx if idx < 8 else idx - 8
    col1 = 1 if idx < 8 else 2
    col2 = 0 if idx < 8 else 3
    qmap[q] = cirq.GridQubit(row, col1)
    qmap[f] = cirq.GridQubit(row, col2)
# pprint(qmap)
mapped_circuit = cirq.Circuit(moment.transform_qubits(qmap) for moment in exp)
mapped_circuit

┌──┐   ┌──┐   ┌────┐   ┌───┐   ┌──┐   ┌─────┐   ┌───┐   ┌───┐   ┌──┐       ┌──┐
(0, 0): ───CULT(0.785)────────────────────────────────────────────────────────────────@────────────────────────────M('C0')─────────────────────────
                                                                                      │
(0, 1): ───R─────────────H─────@─────@─────────@───────@─────@──────────@─────@───────X───────────────────────────────────────S───H───M('q(0)')────
                               │     │         │       │     │          │     │
(0, 2): ───R───────────────────┼─────┼─────────┼─────X─┼─────X──────────┼─────┼───────────────X─────X─────────────────────────S───H───M('q(8)')────
                               │     │         │     │ │                │     │               │     │
(0, 3): ───CULT(0.785)─────────┼─────┼─────────┼─────┼─┼────────────────┼─────┼───────────────┼─────@──────────────M('C8')─────────────────────────
                               │     │         │     │ │                │     │               │
(1, 0): ───CULT(0.785)─────────┼─────┼─────────┼─────┼─┼────────────────┼─────┼───────────────┼@───────────────────M('C1')─────────────────────────
                               │     │         │     │ │                │     │               ││
(1, 1): ───R─────────────H────@┼─────┼@───────@┼─────┼@┼──────@────────@┼─────┼───────@───────┼X──────────────────────────────S───H───M('q(1)')────
                              ││     ││       ││     │││      │        ││     │       │       │
(1, 2): ───R──────────────────┼┼─────┼┼───────┼┼─────┼X┼─────X┼────────┼┼─────┼───────┼X──────┼X──────────────────────────────S───H───M('q(9)')────
                              ││     ││       ││     │ │     ││        ││     │       ││      ││
(1, 3): ───CULT(0.785)────────┼┼─────┼┼───────┼┼─────┼─┼─────┼┼────────┼┼─────┼───────┼┼──────┼@───────────────────M('C9')─────────────────────────
                              ││     ││       ││     │ │     ││        ││     │       ││      │
(2, 0): ───CULT(0.785)────────┼┼─────┼┼───────┼┼─────┼─┼─────┼┼────────┼┼─────┼───────┼┼──────┼────────────────@───M('C2')─────────────────────────
                              ││     ││       ││     │ │     ││        ││     │       ││      │                │
(2, 1): ───R──────────────────┼┼─────┼┼───────┼┼─────┼─┼─────┼X────────┼┼─────X───────┼┼──────┼──────────X─────X──────────────S───H───M('q(2)')────
                              ││     ││       ││     │ │     │         ││             ││      │          │
(2, 2): ───R──────────────────X┼─────X┼───────┼┼─────┼─┼─────┼───────X─┼┼─────X───────┼┼──────┼──────────┼────────────────────S───H───M('q(10)')───
                               │      │       ││     │ │     │       │ ││     │       ││      │          │
(2, 3): ───CULT(0.785)─────────┼──────┼───────┼┼─────┼─┼─────┼───────┼─┼┼─────@───────┼┼──────┼──────────┼─────────M('C10')────────────────────────
                               │      │       ││     │ │     │       │ ││             ││      │          │
(3, 0): ───CULT(0.785)─────────┼──────┼───────┼┼─────┼─┼─────┼───────┼─┼┼─────────────┼┼@─────┼──────────┼─────────M('C3')─────────────────────────
                               │      │       ││     │ │     │       │ ││             │││     │          │
(3, 1): ───R─────────────H────@┼─────@┼──────@┼┼─────┼@┼─────┼@──────┼@┼┼───────@─────┼┼X─────┼──────────┼────────────────────S───H───M('q(3)')────
                              ││     ││      │││     │││     ││      ││││       │     ││      │          │
(3, 2): ───R──────────────────┼┼─────X┼──────┼┼┼─────┼┼┼─────┼┼─────X┼┼┼┼──────X┼─────┼┼X─────┼──────────┼────────────────────S───H───M('q(11)')───
                              ││      │      │││     │││     ││     │││││      ││     │││     │          │
(3, 3): ───CULT(0.785)────────┼┼──────┼──────┼┼┼─────┼┼┼─────┼┼─────┼┼┼┼┼──────┼┼─────┼┼@─────┼──────────┼─────────M('C11')────────────────────────
                              ││      │      

In [5]:
for arc in [
    res.architecture.DualSpeciesMovement(post_op_correction=False, idling=False, cultivation_fault_distance=3, cultivation_repetition=100),
     res.architecture.MeasureZonesOnly(post_op_correction=False, idling=False), 
     res.architecture.DefaultMovement(post_op_correction=False, idling=False),
    ]:
    with_moves = res.compile_ftqc.add_moves(
        mapped_circuit,
        zone_ops=arc.zone_ops if arc.zone_ops is not None else cirq.Gateset(),
        alley_ops=arc.alley_ops if arc.alley_ops is not None else cirq.Gateset(),
    )
    display(with_moves)
    estimator = res.estimate.ResourceEstimator(arc)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        print([str(op.gate) for op in estimator.critical_path(with_moves)])
    _time, _moments, _gates = estimator.parallel_circuit_time(with_moves), estimator.parallel_circuit_cost(with_moves), estimator.serial_circuit_cost(with_moves)
    pprint(_time)
    pprint({GATE2STR[gate]: num for gate, num in _moments.items()})
    pprint({GATE2STR[gate]: num for gate, num in _gates.items()})

┌────────┐   ┌──┐   ┌────────┐   ┌────────┐   ┌──┐   ┌────────┐   ┌────────────────┐   ┌────┐   ┌────────────────┐   ┌────────────┐   ┌───┐   ┌────────────┐   ┌────────┐   ┌──┐   ┌────────┐   ┌────────────────────┐   ┌─────┐   ┌────────────────────┐   ┌────────────┐   ┌───┐   ┌────────────┐   ┌────────────┐   ┌───┐   ┌────────────┐   ┌────────┐   ┌──┐   ┌────────┐                     ┌────────┐   ┌──┐   ┌────────┐
(0, 0): ───CULT(0.785)──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────MOVE─────────────@───────#2────────────────────────────────────────────────────────────────────────────────────────────────────────────────────M('C0')─────────────────────────
                                                                                                                                                                                                                                                                                                                                    │                │       │
(0, 1): ───R─────────────H────────MOVE──────@─────────#2───────MOVE─────────@──────#2───────────────────────MOVE────────@─────────────────#2───────────────MOVE───────@─────────────#2───────MOVE─────────@──────#2───────────────────────────MOVE─────────@─────────────────────#2───────MOVE─────────────@───────#2───────────────#2───────────────X───────MOVE─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────S───H───M('q(0)')────
                                  │         │         │        │            │      │                        │           │                 │                │          │             │        │            │      │                            │            │                     │        │                │       │
(0, 2): ───R──────────────────────┼─────────┼─────────┼────────┼────────────┼──────┼────────────────────────┼───────────┼─────────────────┼────────#2──────┼────────X─┼─────MOVE────┼────────#2───────────X──────MOVE─────────────────────────┼────────────┼─────────────────────┼────────┼────────────────┼───────┼──────────────────────────────────────────────────────────#2───────────X──────MOVE────────#2─────X───MOVE─────────────────────────────────────────────────────────────────S───H───M('q(8)')────
                                  │         │         │        │            │      │                        │           │                 │        │       │        │ │     │       │                                                         │            │                     │        │                │       │                                                          │            │      │           │      │   │
(0, 3): ───CULT(0.785)────────────┼─────────┼─────────┼────────┼────────────┼──────┼────────────────────────┼───────────┼─────────────────┼────────┼───────┼────────┼─┼─────┼───────┼─────────────────────────────────────────────────────────┼────────────┼─────────────────────┼────────┼────────────────┼───────┼──────────────────────────────────────────────────────────┼────────────┼──────┼───────────MOVE───@───#2────────────────────────────────────────────────────────M('C8')─────────────────────────
                                  │         │         │        │            │      │                        │           │                 │        │       │        │ │     │       │                                                         │            │                     │        │                │       │                                                          │            │      │
(1, 0): ───CULT(0.785)────────────┼─────────┼─────────┼────────┼────────────┼──────┼────────────────────────┼───────────┼──

['CULT(0.785)', 'MOVE', 'CNOT', 'MOVE', 'S', 'H', "cirq.MeasurementGate(1, cirq.MeasurementKey(name='q(8)'), ())"]
1740686.62
{'CZ': 10106,
 'MeasurementGate': 1202,
 'PhasedXZGate': 2806,
 'QubitPermutationGate': 5,
 'ResetChannel': 1301}
{'CZ': 2629010,
 'MeasurementGate': 710141,
 'PhasedXZGate': 812445,
 'QubitPermutationGate': 150,
 'ResetChannel': 794272}


┌────────┐   ┌──┐   ┌────────┐   ┌────────┐   ┌──┐   ┌────────┐   ┌────────────────┐   ┌────┐   ┌────────────────┐   ┌────────────┐   ┌───┐   ┌────────────┐   ┌────────┐   ┌──┐   ┌────────┐   ┌────────────────────┐   ┌─────┐   ┌────────────────────┐   ┌────────────┐   ┌───┐   ┌────────────┐   ┌────────────┐   ┌───┐   ┌────────────┐   ┌────────┐   ┌──┐   ┌────────┐                     ┌────────┐   ┌──┐   ┌────────┐
(0, 0): ───CULT(0.785)──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────MOVE─────────────@───────#2────────────────────────────────────────────────────────────────────────────────────────────────────────────────────MOVE_MZ───M('C0')────MOVE_MZ────────────────────────────────────────────
                                                                                                                                                                                                                                                                                                                                    │                │       │
(0, 1): ───R─────────────H────────MOVE──────@─────────#2───────MOVE─────────@──────#2───────────────────────MOVE────────@─────────────────#2───────────────MOVE───────@─────────────#2───────MOVE─────────@──────#2───────────────────────────MOVE─────────@─────────────────────#2───────MOVE─────────────@───────#2───────────────#2───────────────X───────MOVE─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────S───H───MOVE_MZ───M('q(0)')────MOVE_MZ───
                                  │         │         │        │            │      │                        │           │                 │                │          │             │        │            │      │                            │            │                     │        │                │       │
(0, 2): ───R──────────────────────┼─────────┼─────────┼────────┼────────────┼──────┼────────────────────────┼───────────┼─────────────────┼────────#2──────┼────────X─┼─────MOVE────┼────────#2───────────X──────MOVE─────────────────────────┼────────────┼─────────────────────┼────────┼────────────────┼───────┼──────────────────────────────────────────────────────────#2───────────X──────MOVE────────#2─────X───MOVE─────────────────────────────────────────────────────────────────────────────────────S───H───MOVE_MZ───M('q(8)')────MOVE_MZ───
                                  │         │         │        │            │      │                        │           │                 │        │       │        │ │     │       │                                                         │            │                     │        │                │       │                                                          │            │      │           │      │   │
(0, 3): ───CULT(0.785)────────────┼─────────┼─────────┼────────┼────────────┼──────┼────────────────────────┼───────────┼─────────────────┼────────┼───────┼────────┼─┼─────┼───────┼─────────────────────────────────────────────────────────┼────────────┼─────────────────────┼────────┼────────────────┼───────┼──────────────────────────────────────────────────────────┼────────────┼──────┼───────────MOVE───@───#2────────────────────────────────────────────────────────MOVE_MZ───M('C8')────MOVE_MZ────────────────────────────────────────────
                                  │         │         │        │            │      │                        │           │                 │        │       │        │ │     │       │                                                         │            │                     │        │                │       │                                           

['CULT(0.785)', 'MOVE', 'CNOT', 'MOVE', 'S', 'H', 'MOVE_MZ', "cirq.MeasurementGate(1, cirq.MeasurementKey(name='q(8)'), ())", 'MOVE_MZ']
35326.89
{'CZ': 107,
 'MeasurementGate': 14,
 'PhasedXZGate': 34,
 'QubitPermutationGate': 33,
 'ResetChannel': 14}
{'CZ': 31745,
 'MeasurementGate': 9221,
 'PhasedXZGate': 17970,
 'QubitPermutationGate': 598,
 'ResetChannel': 10192}


┌──┐                       ┌──┐                       ┌────┐                       ┌───┐                       ┌──┐                       ┌─────┐                       ┌───┐                       ┌───┐                       ┌──┐                                               ┌──┐
(0, 0): ───CULT(0.785)───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────MOVE_IZ────@──────MOVE_IZ─────────────────────────────────────────────────────────────────────────────────────────────────────────MOVE_MZ───M('C0')────MOVE_MZ────────────────────────────────────────────
                                                                                                                                                                                                                                            │
(0, 1): ───R─────────────H───MOVE_IZ─────@────MOVE_IZ───MOVE_IZ────@─────MOVE_IZ───MOVE_IZ───────@────MOVE_IZ───MOVE_IZ──────@────MOVE_IZ───MOVE_IZ────@─────MOVE_IZ───MOVE_IZ────────@────MOVE_IZ───MOVE_IZ────@──────MOVE_IZ───MOVE_IZ────X──────MOVE_IZ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────S───H───MOVE_MZ───M('q(0)')────MOVE_MZ───
                                         │                         │                             │                           │                         │                              │                         │
(0, 2): ───R─────────────────────────────┼─────────────────────────┼─────────────────────────────┼──────────────MOVE_IZ────X─┼────MOVE_IZ───MOVE_IZ────X─────MOVE_IZ──────────────────┼─────────────────────────┼────────────────────────────────────────────MOVE_IZ────X─────MOVE_IZ───MOVE_IZ───X───MOVE_IZ─────────────────────────────────────────────────────────────────────────────────────S───H───MOVE_MZ───M('q(8)')────MOVE_MZ───
                                         │                         │                             │                         │ │                                                        │                         │                                                       │                         │
(0, 3): ───CULT(0.785)───────────────────┼─────────────────────────┼─────────────────────────────┼─────────────────────────┼─┼────────────────────────────────────────────────────────┼─────────────────────────┼───────────────────────────────────────────────────────┼───────────────MOVE_IZ───@───MOVE_IZ──────────────────────────────────────────────────────MOVE_MZ───M('C8')────MOVE_MZ────────────────────────────────────────────
                                         │                         │                             │                         │ │                                                        │                         │                                                       │
(1, 0): ───CULT(0.785)───────────────────┼─────────────────────────┼─────────────────────────────┼─────────────────────────┼─┼────────────────────────────────────────────────────────┼─────────────────────────┼────────────────────────────────────────────MOVE_IZ────┼@────MOVE_IZ──────────────────────────────────────────────────────────────────────────────MOVE_MZ───M('C1')────MOVE_MZ────────────────────────────────────────────
                                         │                         │                             │                         │ │                                                        │                         │                                                       ││
(1, 1): ───R─────────────H───MOVE_IZ────@┼────MOVE_IZ───MOVE_IZ────┼@────MOVE_IZ───MOVE_IZ──────@┼────MOVE_IZ───MOVE_IZ────┼@┼────MOVE_IZ───MOVE_IZ─────@────MOVE_IZ───MOVE_IZ───────@┼────MOVE_IZ──────────────┼────────────────MOVE_IZ────@──────MOVE_IZ───MOVE_IZ────┼X────MOVE_IZ──────────────────────────────

['CULT(0.785)', 'MOVE_IZ', 'CNOT', 'MOVE_IZ', 'S', 'H', 'MOVE_MZ', "cirq.MeasurementGate(1, cirq.MeasurementKey(name='q(8)'), ())", 'MOVE_MZ']
141298.89
{'CZ': 107,
 'MeasurementGate': 14,
 'PhasedXZGate': 34,
 'QubitPermutationGate': 243,
 'ResetChannel': 14}
{'CZ': 31745,
 'MeasurementGate': 9221,
 'PhasedXZGate': 17970,
 'QubitPermutationGate': 3848,
 'ResetChannel': 10192}
